In [ ]:
import imageio
from PIL import Image, ImageDraw, ImageFont
import numpy as np

# ---------------- SETTINGS ----------------
gif_paths = [
    "induction_equation_1_year_v_mer7.gif",
    "induction_equation_1_year_v_mer10.gif",
    "induction_equation_1_year_v_mer15.gif",
    "induction_equation_1_year_v_mer20.gif",
]

captions = [
    r"$v_{\mathrm{mer}} = 7$ m/s",
    r"$v_{\mathrm{mer}} = 10$ m/s",
    r"$v_{\mathrm{mer}} = 15$ m/s",
    r"$v_{\mathrm{mer}} = 20$ m/s",
]

output_mp4 = "combined_square_vmer_with_captions.mp4"

target_width = 400
caption_height = 50
fps = 2
# ------------------------------------------

# Font
try:
    font = ImageFont.truetype("DejaVuSans.ttf", 22)
except:
    font = ImageFont.load_default()

# Load GIFs
gif_frames = []
gif_durations = []

for path in gif_paths:
    reader = imageio.get_reader(path)
    frames = []
    durations = []

    for frame in reader:
        frames.append(Image.fromarray(frame))
        durations.append(reader.get_meta_data().get("duration", 50) / 1000)

    gif_frames.append(frames)
    gif_durations.append(sum(durations))

# Sync duration
max_dur = max(gif_durations)
num_frames = int(max_dur * fps)

# Resize and add captions
processed_sequences = []

for frames, caption in zip(gif_frames, captions):
    seq = []
    for frame in frames:
        w, h = frame.size
        scale = target_width / w
        new_h = int(h * scale)

        frame = frame.resize((target_width, new_h), Image.LANCZOS)

        canvas = Image.new("RGB", (target_width, new_h + caption_height), "white")
        canvas.paste(frame, (0, 0))

        draw = ImageDraw.Draw(canvas)
        text_w, text_h = draw.textsize(caption, font=font)

        draw.text(
            ((target_width - text_w) // 2, new_h + (caption_height - text_h) // 2),
            caption,
            fill="black",
            font=font
        )

        seq.append(canvas)

    processed_sequences.append(seq)

# Build square video
writer = imageio.get_writer(output_mp4, fps=fps)

for i in range(num_frames):
    frames = [seq[i % len(seq)] for seq in processed_sequences]

    top = np.hstack([np.array(frames[0]), np.array(frames[1])])
    bottom = np.hstack([np.array(frames[2]), np.array(frames[3])])
    combined = np.vstack([top, bottom])

    writer.append_data(combined)

writer.close()
print("Saved:", output_mp4)
